# Thu thập và gán nhãn bình luận PTIT


Trình bày một đợt thu thập bằng **mã và snapshot thật đã lưu**. Luồng:
Facebook → bài/comment/reply → JSON → bảng → clean → gợi ý/chốt nhãn → CSV có nhãn.
Phục vụ ý 1 của đề, phần clean thuộc ý 2. Run All không đăng nhập Facebook, không
gọi LLM/Argilla; ví dụ nhãn dùng cache/export đã có và giữ nguyên ID lịch sử.

Notebook 02 dùng **dataset v7 đã chốt** cho thực nghiệm. Notebook này không nhận
đầu ra minh họa là toàn bộ v7. Thông tin nguồn/phiên bản: `docs/provenance/`.


## 0. Môi trường và đường dẫn

Chạy trong repo với môi trường `requirements/notebook.txt`. Snapshot chưa có sẽ
được tải từ Drive theo manifest và kiểm tra SHA-256, không cần cookie Facebook.


In [1]:
import os, sys
from pathlib import Path

# Mở notebook trong repo; không cần nhúng lại module bằng %%writefile.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "pipeline/__init__.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Mở notebook trong thư mục project có pipeline/ và external/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import ast, hashlib, json
import pandas as pd
from IPython.display import Code, display
from pipeline.collect.json_to_csv import flatten_posts
from pipeline.collect.snapshots import ensure_snapshots
from pipeline.prepare.clean_for_labeling import preprocess as clean_for_labeling
from pipeline.label.label_utils import join_metadata

SOURCE_ROOT = Path(os.environ.get("SPAMHAM_SOURCE_DIR", str(PROJECT_ROOT / "data")))
ARTIFACT_DIR = Path(os.environ.get("SPAMHAM_OUTPUT_DIR", str(PROJECT_ROOT / "artifacts/collection")))
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 100)

def show_code(relative_path, function_name, max_lines=35):
    # Đọc AST để show đoạn code thực, không import/chạy module thu thập.
    source = (PROJECT_ROOT / relative_path).read_text(encoding="utf-8")
    node = next(n for n in ast.walk(ast.parse(source))
                if isinstance(n, (ast.FunctionDef, ast.ClassDef)) and n.name == function_name)
    lines = ast.get_source_segment(source, node).splitlines()
    print(f"{relative_path}:{node.lineno} — {function_name}; hiển thị {min(len(lines), max_lines)}/{len(lines)} dòng")
    display(Code("\n".join(lines[:max_lines]), language="python"))


## 1. Nguồn thu thập và thư viện sử dụng — ý 1

`pipeline/collect/scrape_ptit_groups.py` là runner PTIT. Các module nền được giữ ở
`external/facebook_scraper/`, dựa trên repo
https://github.com/mohdtalal3/facebook_post_comment_scraper . Bản bàn giao có một
số module đã thay đổi, không gọi toàn bộ là mã nhóm tự viết hoặc upstream nguyên xi.


In [2]:
source_names = ["groups_data_phase2.json", "comments_phase2.csv", "comments_phase2_cleaned.csv",
                "predictions_glm_cache_phase2.json", "comments_labeled_phase2.csv"]
snapshot_paths = ensure_snapshots(SOURCE_ROOT, source_names)
posts = json.loads(snapshot_paths["groups_data_phase2.json"].read_text(encoding="utf-8-sig"))
groups = pd.DataFrame([{"group_id": p["group_id"], "group_name": p["group_name"]} for p in posts])
group_table = groups.groupby(["group_id", "group_name"]).size().rename("số bài trong snapshot").reset_index()
group_table["link từ ID"] = "https://www.facebook.com/groups/" + group_table.group_id.astype(str) + "/"
display(group_table)
print("Snapshot đã lưu:", len(posts), "bài; số group:", len(group_table))


,group_id,group_name,số bài trong snapshot,link từ ID
0,1130813927934700,D26 (2008) PTIT - Học viện Công nghệ Bưu chính Viễn thông,500,https://www.facebook.com/groups/1130813927934700/
1,1210825580901941,PTIT Confessions - Sinh Viên Học Viện Công Nghệ Bưu Chính Viễn Thông ✅,500,https://www.facebook.com/groups/1210825580901941/
2,1605563914144178,D26 Học Viện Công Nghệ Bưu Chính Viễn Thông (2K8 PTIT),500,https://www.facebook.com/groups/1605563914144178/
3,450310163844833,Cộng đồng sinh viên PTIT,500,https://www.facebook.com/groups/450310163844833/
4,778910719727961,D21 PTIT Hà Nội,170,https://www.facebook.com/groups/778910719727961/


Snapshot đã lưu: 2170 bài; số group: 5


## 2. Code lấy bài, comment/reply và lưu JSON — ý 1

Runner đặt group/context → `fetch_posts()` → callback lấy bình luận bằng
`fetch_comments_for_post()` → `fetch_comments()`/`fetch_replies()` → writer.
Đoạn code dưới đọc trực tiếp từ file thật. JSON có group/post ID, permalink,
thời gian bài, nội dung và comments/replies; chưa có nhãn spam/ham.

Snapshot không chứng minh đã lấy hết mọi comment/reply. Phân trang replies và
checkpoint/resume còn giới hạn. Chạy live cần kiểm tra phiên/quyền/endpoint riêng.


In [3]:
show_code("pipeline/collect/scrape_ptit_groups.py", "on_batch", 30)
show_code("external/facebook_scraper/main.py", "fetch_comments_for_post", 35)
show_code("pipeline/collect/scrape_ptit_groups.py", "save_records", 30)
example = next(p for p in posts if p.get("comments"))
display(pd.Series({key: example.get(key) for key in ["group_id", "group_name", "post_id", "permalink", "created_at", "scraped_at"]}).to_frame("record thật"))
print("Các trường JSON:", sorted(example))


pipeline/collect/scrape_ptit_groups.py:475 — on_batch; hiển thị 30/49 dòng


def on_batch(batch_posts, total_so_far, total_limit):
            nonlocal group_records_count, total_scraped_all_groups
            processed_records = []

            for post in batch_posts:
                post_id = post.get("post_id")
                if not post_id:
                    continue

                if writer.is_already_saved(post_id):
                    log_message(f"  ⏭️ Bỏ qua bài đã có trong file: {post_id}")
                    continue

                comments = []
                comment_texts = []
                if not args.no_comments:
                    try:
                        comments, _ = fetch_comments_for_post(post_id, cookies=cookies)
                        comment_texts = extract_comment_texts(comments)
                        if comments_writer.output_path and comment_texts:
                            comments_writer.append_comments(group_name, post_id, comment_texts)
                        time.sleep(args.delay)
                    except FacebookSessionExpiredException:
                        raise  # Chuyển tiếp ngoại lệ session hết hạn lên cấp xử lý chính
                    except Exception as e:
                        log_message(f"  ⚠️ Lỗi lấy comment bài {post_id}: {e}")

                record = {
                    "group_id": group_id,
                    "group_name": group_name,

external/facebook_scraper/main.py:211 — fetch_comments_for_post; hiển thị 22/22 dòng


def fetch_comments_for_post(post_id, cookies=None):
    """Fetch all comments and replies for a given post_id"""
    feedback_id = convert_post_id_to_feedback_id(post_id)
    print(f"  Fetching comments for post {post_id}...")
    print(f"  Using feedback_id: {feedback_id}")
    
    all_data = []
    comments, post_info = fetch_comments(feedback_id, cookies=cookies)
    
    for c in comments:
        print(f"    🗨️ {c.get('text', '')[:50]}...")
        c["replies"] = fetch_replies(c, cookies=cookies)
        
        for r in c["replies"]:
            print(f"       ↳ {r.get('text', '')[:50]}...")
        
        # Remove internal fields before appending
        c_clean = {k: v for k, v in c.items() if not k.startswith('_')}
        all_data.append(c_clean)
    
    print(f"  ✓ Found {len(all_data)} comments")
    return all_data, post_info

pipeline/collect/scrape_ptit_groups.py:311 — save_records; hiển thị 21/21 dòng


def save_records(self, new_records):
        """Lưu danh sách bài viết mới vào file duy nhất"""
        if not new_records:
            return

        for record in new_records:
            pid = str(record.get("post_id"))
            self.existing_post_ids.add(pid)

        if self.is_jsonl:
            with open(self.output_path, "a", encoding="utf-8") as f:
                for r in new_records:
                    f.write(json.dumps(r, ensure_ascii=False) + "\n")
                f.flush()
        else:
            self.all_records.extend(new_records)
            # Ghi ra file tạm rồi đổi tên (atomic write) để đảm bảo file JSON không bao giờ bị hỏng nếu bị ngắt tiến trình
            temp_path = self.output_path + ".tmp"
            with open(temp_path, "w", encoding="utf-8") as f:
                json.dump(self.all_records, f, ensure_ascii=False, indent=2)
            os.replace(temp_path, self.output_path)

,record thật
group_id,778910719727961
group_name,D21 PTIT Hà Nội
post_id,2015992686019752
permalink,https://www.facebook.com/groups/d21ptithanoii/posts/2015992686019752/
created_at,2026-03-25 17:35:43
scraped_at,2026-09-27 10:06:57


Các trường JSON: ['comment_count', 'comment_texts', 'comments', 'created_at', 'creation_time', 'group_id', 'group_name', 'message', 'permalink', 'photos', 'post_id', 'scraped_at', 'videos']


## 3. JSON → mỗi comment/reply một dòng CSV — chuẩn bị dữ liệu

`pipeline/collect/json_to_csv.py: flatten_posts()` giữ metadata và tách comment/reply.
Ở đây chạy trên **3 bài có bình luận** để nhìn rõ phép biến đổi. Đây là CSV xem thử,
không phải toàn bộ CSV lịch sử và không dùng chỉ số dòng này làm ID cache cũ.
Converter CLI còn có lọc rỗng/trùng với dữ liệu tham chiếu tùy cấu hình.


In [4]:
preview_posts = [p for p in posts if p.get("comments")][:3]
preview = pd.DataFrame(flatten_posts(preview_posts))
assert "label" not in preview
display(preview.head(8))
preview.to_csv(ARTIFACT_DIR / "comments_preview.csv", index=False, encoding="utf-8-sig")
show_code("pipeline/collect/json_to_csv.py", "flatten_posts", 35)


,post_id,group_id,group_name,created_at,comment_type,text,reaction_count
0,2015992686019752,778910719727961,D21 PTIT Hà Nội,2026-03-25 17:35:43,comment,Huy Hoàng Nguyễn Hữu Thắng người thầy tôi yêu,1
1,2015992686019752,778910719727961,D21 PTIT Hà Nội,2026-03-25 17:35:43,comment,"Cô Chi nhé, bao điểm cao luôn",0
2,2015992686019752,778910719727961,D21 PTIT Hà Nội,2026-03-25 17:35:43,comment,Thầy hùng nhé,0
3,2015992686019752,778910719727961,D21 PTIT Hà Nội,2026-03-25 17:35:43,reply,Con_cá_cắn_cáp thầy Hùng bên cntt hay thầy Phạm Hùng bên viễn thông ạ,0
4,2015992686019752,778910719727961,D21 PTIT Hà Nội,2026-03-25 17:35:43,comment,"Thầy Kiên nha em, thầy 10 điểm ko có nhưng",0
5,2015992686019752,778910719727961,D21 PTIT Hà Nội,2026-03-25 17:35:43,comment,Cô thư điểm cao,0
6,2179604596325226,778910719727961,D21 PTIT Hà Nội,2026-09-14 11:57:26,comment,Chụp ảnh ib anh tư vấn nhe,0
7,2179604596325226,778910719727961,D21 PTIT Hà Nội,2026-09-14 11:57:26,reply,Nông Thành Long có đưa ảnh em thì dùng cái pts rồi nghe chưa :)))),1


pipeline/collect/json_to_csv.py:39 — flatten_posts; hiển thị 25/25 dòng


def flatten_posts(posts: list) -> list:
    """post -> comments -> replies  ==>  danh sách dòng phẳng."""
    rows = []
    for post in posts:
        base = {
            "post_id": post.get("post_id", ""),
            "group_id": post.get("group_id", ""),
            "group_name": post.get("group_name", ""),
            "created_at": post.get("created_at", ""),
        }
        for com in post.get("comments", []) or []:
            rows.append({
                **base,
                "comment_type": "comment",
                "text": (com.get("text") or "").strip(),
                "reaction_count": to_int(com.get("reaction_count")),
            })
            for rep in com.get("replies", []) or []:
                rows.append({
                    **base,
                    "comment_type": "reply",
                    "text": (rep.get("text") or "").strip(),
                    "reaction_count": to_int(rep.get("reaction_count")),
                })
    return rows

## 4. Clean trước gán nhãn — ý 2

Giữ `text` gốc và thêm `cleaned_text`. Clean này chuẩn bị bảng để gán nhãn; chưa
phải tách từ/đối chứng stop words cho NB/SVM ở notebook 02. Cleaner CLI còn khử
trùng/lọc dòng. Không đổi thứ tự snapshot rồi dùng lại cache ID theo vị trí cũ.


In [5]:
preview_clean = preview.copy()
preview_clean["cleaned_text"] = preview_clean.text.map(clean_for_labeling)
display(preview_clean[["text", "cleaned_text"]].head(8))
preview_clean.to_csv(ARTIFACT_DIR / "cleaned_preview.csv", index=False, encoding="utf-8-sig")
show_code("pipeline/prepare/clean_for_labeling.py", "preprocess", 32)


,text,cleaned_text
0,Huy Hoàng Nguyễn Hữu Thắng người thầy tôi yêu,Huy Hoàng Nguyễn Hữu Thắng người thầy tôi yêu
1,"Cô Chi nhé, bao điểm cao luôn","Cô Chi nhé, bao điểm cao luôn"
2,Thầy hùng nhé,Thầy hùng nhé
3,Con_cá_cắn_cáp thầy Hùng bên cntt hay thầy Phạm Hùng bên viễn thông ạ,Con_cá_cắn_cáp thầy Hùng bên cntt hay thầy Phạm Hùng bên viễn thông ạ
4,"Thầy Kiên nha em, thầy 10 điểm ko có nhưng","Thầy Kiên nha em, thầy 10 điểm ko có nhưng"
5,Cô thư điểm cao,Cô thư điểm cao
6,Chụp ảnh ib anh tư vấn nhe,Chụp ảnh ib anh tư vấn nhe
7,Nông Thành Long có đưa ảnh em thì dùng cái pts rồi nghe chưa :)))),Nông Thành Long có đưa ảnh em thì dùng cái pts rồi nghe chưa :))))


pipeline/prepare/clean_for_labeling.py:89 — preprocess; hiển thị 32/32 dòng


def preprocess(text: str) -> str:
    """
    Tiền xử lý văn bản tiếng Việt cho bài toán Spam/Ham.
    Trả về text_clean (URL/SĐT được thay bằng token, không xóa trắng).
    """
    if not text:
        return ""

    # 1. Unicode NFC
    text = unicodedata.normalize("NFC", text)
    # 2. Decode HTML entities
    text = html.unescape(text)
    # 3. Xóa thẻ HTML
    text = _RE_HTML_TAG.sub(" ", text)

    # 4. Thay URL bằng token (KHÔNG xóa trắng - URL là tín hiệu spam quan trọng)
    text = _RE_URL.sub(" urltoken ", text)

    # 5. Thay số điện thoại VN bằng token (tín hiệu spam quan trọng)
    text = _RE_PHONE_VN.sub(" phonetoken ", text)

    # 6. Xóa emoji & ký tự hình ảnh
    text = _RE_EMOJI.sub(" ", text)
    # 7. Xóa ký tự đặc biệt, giữ lại chữ cái, số, dấu câu (urltoken/phonetoken vẫn giữ được)
    text = _RE_KEEP.sub(" ", text)
    # 8. Chuẩn hóa dấu câu lặp
    text = _RE_PUNCT_REP.sub(r"\1", text)
    text = _RE_DOTS_REP.sub("...", text)
    # 9. Chuẩn hóa khoảng trắng
    text = _RE_SPACE.sub(" ", text)

    return text.strip()

## 5. Gợi ý AI → chấp nhận/duyệt → export nhãn

| File trong `pipeline/label/` | Nhiệm vụ |
|---|---|
| `upload_to_argilla.py` | Upload văn bản và câu hỏi nhãn |
| `ai_assisted_spam_ham.py` | Gợi ý label/confidence, lưu cache; tùy chọn upload/tự chốt |
| `auto_accept_labels.py` | Chấp nhận theo ngưỡng cấu hình; không đồng nghĩa đã kiểm tay |
| `download_labeled_data.py` | Export response submitted, nối metadata |
| `label_utils.py` | Kiểm tra đúng ID/text, chuẩn hóa dự đoán |

Cell tiếp theo **đọc cache/export đã lưu**, không gọi AI hoặc Argilla. Dùng ID của
bảng cleaned lịch sử để nối đúng comment. Confidence là điểm do AI cung cấp,
không phải tỷ lệ đúng đã đo. Lịch sử người duyệt chưa được chứng minh đầy đủ.


In [6]:
read_snapshot_csv = lambda name: pd.read_csv(snapshot_paths[name], dtype=str, keep_default_na=False)
historical_clean = read_snapshot_csv("comments_phase2_cleaned.csv")
official = read_snapshot_csv("comments_labeled_phase2.csv")
cache = json.loads(snapshot_paths["predictions_glm_cache_phase2.json"].read_text(encoding="utf-8"))
sample = official[official.id.isin(cache)].head(8).copy()
sample = join_metadata(sample, historical_clean)
sample["gợi ý đã lưu"] = sample.id.map(lambda rid: cache[rid]["label"])
sample["confidence đã lưu"] = sample.id.map(lambda rid: cache[rid]["confidence"])
display(sample[["id", "text", "gợi ý đã lưu", "confidence đã lưu", "label"]])
show_code("pipeline/label/ai_assisted_spam_ham.py", "predict_batch_with_glm", 28)
show_code("pipeline/label/download_labeled_data.py", "extract_submitted_response", 22)


,id,text,gợi ý đã lưu,confidence đã lưu,label
0,cmt_0,Huy Hoàng Nguyễn Hữu Thắng người thầy tôi yêu,ham,0.95,ham
1,cmt_1,"Cô Chi nhé, bao điểm cao luôn",ham,0.95,ham
2,cmt_2,Thầy hùng nhé,ham,0.95,ham
3,cmt_3,Con_cá_cắn_cáp thầy Hùng bên cntt hay thầy Phạm Hùng bên viễn thông ạ,ham,0.95,ham
4,cmt_4,"Thầy Kiên nha em, thầy 10 điểm ko có nhưng",ham,0.95,ham
5,cmt_5,Cô thư điểm cao,ham,0.95,ham
6,cmt_6,Chụp ảnh ib anh tư vấn nhe,ham,0.80,ham
7,cmt_7,Nông Thành Long có đưa ảnh em thì dùng cái pts rồi nghe chưa :)))),ham,0.95,ham


pipeline/label/ai_assisted_spam_ham.py:230 — predict_batch_with_glm; hiển thị 28/101 dòng


def predict_batch_with_glm(batch_items, api_key: str, model: str, base_url: str, max_retries: int = 3):
    """Gửi 1 batch đánh giá tới endpoint tương thích OpenAI bằng mô hình GLM."""
    prompt_lines = ["Dưới đây là danh sách các bình luận cần phân loại spam/ham:\n"]
    for item in batch_items:
        prompt_lines.append(f'- ID: {item["id"]} | Nội dung: "{item["text"]}"')

    prompt_content = "\n".join(prompt_lines)

    endpoint_url = base_url.rstrip("/")
    if not endpoint_url.endswith("/chat/completions"):
        endpoint_url = f"{endpoint_url}/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    if "openrouter" in base_url.lower():
        headers["HTTP-Referer"] = "http://localhost:6900"
        headers["X-Title"] = "Argilla Spam/Ham AI Annotation"

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt_content},
        ],
        "temperature": 0.1,
        "max_tokens": 3500,

pipeline/label/download_labeled_data.py:95 — extract_submitted_response; hiển thị 8/8 dòng


def extract_submitted_response(rec):
    """Lấy response có trạng thái 'submitted' từ một bản ghi (lấy phần tử cuối trong danh sách; không suy diễn thứ tự thời gian)."""
    submitted = [
        r for r in getattr(rec, "responses", []) or [] if getattr(r, "status", "") == "submitted"
    ]
    if not submitted:
        return None
    return submitted[-1]

## 6. CSV có nhãn và chuyển sang thực nghiệm

Xuất một bảng nhãn xem thử từ **các record chính thức đã lưu**, không tự tạo nhãn
mới. Toàn bộ export của đợt này có số dòng riêng; nó không được nhận là v7.
Notebook 02 bắt đầu từ `data/final/labeled_v7.csv` do nhóm cung cấp.

Công cụ gộp/đối soát ở `tools/dataset_versions/`; nguồn và giới hạn phiên bản ở
`docs/provenance/`. Không cần chạy chúng để notebook 02 dùng v7.


In [7]:
columns = ["id", "label", "cleaned_text", "text", "comment_type", "reaction_count"]
sample[columns].to_csv(ARTIFACT_DIR / "labeled_preview.csv", index=False, encoding="utf-8-sig")
source_hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in snapshot_paths.items()}
manifest = {"mode": "saved_real_snapshot", "facebook_requests": False, "llm_argilla_calls": False,
            "raw_posts": len(posts), "groups": len(group_table), "preview_posts": len(preview_posts),
            "preview_rows": len(preview), "official_export_rows": len(official),
            "labeled_preview_rows": len(sample), "source_sha256": source_hashes,
            "next_notebook": "02_thuc_nghiem_va_demo.ipynb"}
(ARTIFACT_DIR / "collection_demo_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã tạo CSV xem thử và manifest tại", ARTIFACT_DIR)


Đã tạo CSV xem thử và manifest tại C:\KH086\PT&KP DLVB\PROJECT\ptit-facebook-spam-ham\.work\runs\restructured_collection_release\artifacts
